# 票据可信度核验 · CORD 公开数据集实验

工行杯 · 金融安全服务方向

三个实验：

| | 测什么 | 指标 |
|---|---|---|
| E1 | 公开票据本身有多少是自洽的 | 误报基线 |
| E2 | 模型读出的字段与标注是否一致 | 各字段精确匹配率 |
| E3 | 被篡改的票据能否被发现 | TPR / FPR |

**运行环境**：Colab CPU 即可。模型在服务端，GPU 用不上。
需要 Colab Secrets 里有 `DEEPSEEK_API_KEY`。


## 1. 装依赖与配置 API

In [ ]:
!pip install -q datasets langchain-core langchain-deepseek

import os
from pathlib import Path
try:
    from google.colab import userdata
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
except Exception as e:
    raise SystemExit(f"请先在左栏钥匙图标里加 DEEPSEEK_API_KEY：{e}")

Path("/content/ghb/src").mkdir(parents=True, exist_ok=True)
print("API key 已就绪")

## 2. 写入源码

与本地 `ghb/src/` 保持一致，改动后重跑 `build_notebooks.py` 即可同步。

In [ ]:
%%writefile /content/ghb/src/money.py
"""金额归一化。

票据金额的书写方式跨地区差异很大，同一个数据集内部也不统一。CORD（印尼）里
同时出现 "60.000"、"91000"、"28,000"、"Rp. 111,000"；港式票据用 "$316.10"。
若按英文习惯把 "60.000" 读成 60.00，六万会变成六十，后面所有校验都失去意义。

规则：
  1. 去掉货币符号与非数字尾巴
  2. 只剩一种分隔符且最后一段是 3 位 -> 千位分隔符，全部删去
  3. 同时出现 "." 和 ","        -> 最后出现的那个是小数点
  4. 只剩一种分隔符且最后一段不是 3 位 -> 小数点
"""
import re
from decimal import Decimal, InvalidOperation

# 允许 "Rp. 111,000"、"HK$1,234.50"、"(5.39)"、"-$16.59"
_NUM = re.compile(r"-?\d[\d.,]*")
_CURRENCY = re.compile(r"(?i)\b(?:rp|hk|idr|usd|sgd|myr|php|thb)\b\.?\s*")


def parse_amount(value, thousands_hint=None):
    """把票据上的金额写法转成 Decimal；无法解析时返回 None。

    thousands_hint: "." 或 "," 可强制指定千位分隔符（已知地区时更稳）。
    """
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, (int, Decimal)):
        return Decimal(str(value))
    if isinstance(value, float):
        return Decimal(str(value))
    if not isinstance(value, str):
        return None

    text = _CURRENCY.sub("", value).replace("$", "").replace("￥", "").replace("¥", "")
    text = text.strip()
    negative = text.startswith("(") and text.endswith(")")
    text = text.strip("()").strip()

    m = _NUM.search(text)
    if not m:
        return None
    raw = m.group(0)
    neg = negative or raw.startswith("-")
    raw = raw.lstrip("-").rstrip(".,")
    if not raw:
        return None

    has_dot, has_comma = "." in raw, "," in raw
    if has_dot and has_comma:
        # 两种都有，最后出现的是小数点
        dec_sep = "." if raw.rfind(".") > raw.rfind(",") else ","
        thou_sep = "," if dec_sep == "." else "."
        raw = raw.replace(thou_sep, "").replace(dec_sep, ".")
    elif has_dot or has_comma:
        sep = "." if has_dot else ","
        if thousands_hint == sep:
            raw = raw.replace(sep, "")
        else:
            tail = raw.rsplit(sep, 1)[1]
            groups = raw.split(sep)
            # 每段都是 3 位且不止一段 -> 千位分隔符（"1.234.567"）
            if len(tail) == 3 and all(len(g) == 3 for g in groups[1:]):
                raw = raw.replace(sep, "")
            else:
                raw = raw.replace(sep, ".")

    try:
        amount = Decimal(raw)
    except InvalidOperation:
        return None
    return -amount if neg else amount


def q2(value):
    """量化到两位小数，便于比较。"""
    return None if value is None else value.quantize(Decimal("0.01"))


In [ ]:
%%writefile /content/ghb/src/receipt.py
"""统一的校验层：把模型的转写折算成可判定的量，跑三条校验，再跨多次识别投票。

三条校验全部在这里执行，从不进入提示词。模型不知道自己被什么标准检查，
也就无法朝那个标准编数字。
"""
import re
from decimal import Decimal

from money import parse_amount

ZERO = Decimal("0.00")


def _amounts(raw, hint):
    """把一列金额归一化成正数 Decimal。"""
    if isinstance(raw, (int, float, str, Decimal)):
        raw = [raw]
    if not isinstance(raw, list):
        return []
    out = []
    for e in raw:
        if isinstance(e, dict):
            e = e.get("amount", e.get("value", e.get("price")))
        v = parse_amount(e, hint)
        if v is not None and v != 0:
            out.append(abs(v))
    return out


def _label_confirms(label, amount):
    """标签文字里是否重复印了这笔金额。

    港式折扣行常把金额写进标签本身（"Buy 3 Save $9.8"），一个数字印了两遍，
    标签与金额栏互为佐证。模型只被要求照抄标签，并不知道这里在比对。
    """
    if not isinstance(label, str):
        return False
    for m in re.finditer(r"\d+(?:[.,]\d+)?", label):
        v = parse_amount(m.group(0))
        if v is not None and abs(v) == amount:
            return True
    return False


def _discounts(raw, rounding, hint):
    """返回 (折扣金额列表, 标签印证条数)。"""
    if isinstance(raw, (int, float, str, Decimal)):
        raw = [raw]
    if not isinstance(raw, list):
        return [], 0
    amounts, confirmed = [], 0
    for e in raw:
        label = None
        if isinstance(e, dict):
            label = e.get("label", e.get("text", e.get("description")))
            e = e.get("amount", e.get("value", e.get("price")))
        v = parse_amount(e, hint)
        if v is None or v == 0:
            continue
        v = abs(v)
        # 舍入行被误放进折扣里时可识别：它等于 |rounding|
        if rounding != 0 and v == abs(rounding):
            continue
        amounts.append(v)
        confirmed += _label_confirms(label, v)
    return amounts, confirmed


def parse_reading(payload, hint=None):
    """把一次转写折算成校验所需的量；无法解析返回 None。"""
    if not isinstance(payload, dict):
        return None
    sub = parse_amount(payload.get("subtotal"), hint)
    total = parse_amount(payload.get("total_paid"), hint)
    if sub is None and total is None:
        return None
    rounding = parse_amount(payload.get("rounding"), hint) or ZERO
    tax = parse_amount(payload.get("tax"), hint) or ZERO
    service = parse_amount(payload.get("service"), hint) or ZERO
    discounts, confirmed = _discounts(payload.get("discounts"), rounding, hint)
    items = _amounts(payload.get("items"), hint)
    disc_total = sum(discounts, ZERO)
    items_total = sum(items, ZERO)
    if sub is None:
        sub = total - tax - service + disc_total - rounding
    if total is None:
        total = sub + tax + service - disc_total + rounding
    return {
        "items_total": items_total, "n_items": len(items),
        "discount_total": disc_total, "labels_ok": confirmed,
        "subtotal": sub, "tax": tax, "service": service,
        "rounding": rounding, "total_paid": total,
    }


# 「小计」的语义跨地区不同，这是实务里必须配置的参数，不该靠猜：
#   港式超市   SUBTOTAL 印的是折扣「后」金额  -> discount_in_subtotal=True
#   印尼 CORD  subtotal_price 是折扣「前」金额 -> discount_in_subtotal=False
# 同一条公式套两地必错一头，所以把它显式化。
HK = {"discount_in_subtotal": True}
IDR = {"discount_in_subtotal": False}


def checks(rec, tol=Decimal("0.05"), locale=HK):
    """三条校验的差额。返回 None 表示这条校验在这张票上不适用。"""
    in_sub = locale["discount_in_subtotal"]
    # 校验一：实付 = 小计 + 税 + 服务费 + 舍入（小计若为折扣前，还要减去折扣）
    expect_total = (rec["subtotal"] + rec["tax"] + rec["service"] + rec["rounding"]
                    - (ZERO if in_sub else rec["discount_total"]))
    c1 = rec["total_paid"] - expect_total
    # 校验二：商品行总额（小计若为折扣后，要减去折扣）应等于小计
    c2 = None
    if rec["n_items"]:
        expect_sub = rec["items_total"] - (rec["discount_total"] if in_sub else ZERO)
        c2 = expect_sub - rec["subtotal"]
    return {
        "c1_gap": c1, "c1_pass": abs(c1) <= tol,
        "c2_gap": c2, "c2_pass": None if c2 is None else abs(c2) <= tol,
        "labels_ok": rec["labels_ok"],
    }


def verdict(rec, tol=Decimal("0.05"), locale=HK):
    """可信 / 存疑，以及失败的是哪几条。"""
    c = checks(rec, tol, locale)
    failed = []
    if not c["c1_pass"]:
        failed.append("c1")
    if c["c2_pass"] is False:
        failed.append("c2")
    return ("存疑" if failed else "可信"), failed, c


def merge(readings, tol=Decimal("0.05"), locale=HK):
    """跨多次识别取共识。

    证据强弱：自洽的优先 -> 标签印证多的优先 -> 多数投票。
    单次识别的随机误差在这里被消掉，而排序依据全部来自代码侧。
    """
    from collections import Counter

    recs = [r for r in readings if r]
    if not recs:
        return None
    ok = [r for r in recs if checks(r, tol, locale)["c1_pass"]] or recs
    consistent = [r for r in ok if checks(r, tol, locale)["c2_pass"] is not False] or ok
    best = max(r["labels_ok"] for r in consistent)
    pool = [r for r in consistent if r["labels_ok"] == best]

    out = {}
    for f in ("subtotal", "tax", "service", "rounding", "total_paid",
              "discount_total", "items_total"):
        vals = [r[f] for r in pool]
        v, n = Counter(vals).most_common(1)[0]
        out[f] = v if n > 1 or len(pool) == 1 else pool[0][f]
    out["n_items"] = pool[0]["n_items"]
    out["labels_ok"] = best
    out["reads"] = len(recs)
    out["pool"] = len(pool)
    return out


In [ ]:
%%writefile /content/ghb/src/chain.py
"""视觉模型链路：prompt -> 模型 -> JSON。

模型只做转写。提示词里没有任何等式——告诉模型的约束，模型就有动力去满足它
而不是读准，这是早期版本踩过的坑（见 docs/项目总纲.md）。

默认接 DeepSeek 视觉模型，换别家只需改 MODEL 与 build_chain 里的一行。
"""
import base64
import mimetypes
import os
from pathlib import Path

MODEL = os.environ.get("RECEIPT_MODEL", "deepseek-v4-flash-vision-exp")
CONCURRENCY = int(os.environ.get("RECEIPT_CONCURRENCY", 5))

SYSTEM_PROMPT = """你在转写一张零售票据。照抄票面上印的内容，不要做任何计算。

按 JSON 输出这些字段：
- "items"：每一行会让账单变大的金额——商品行、包装费、押金、服务费行——一行一条，正数，按票面顺序。
- "discounts"：每一行会让账单变小的金额——折扣、促销、优惠券、"% OFF"、"MEMBER PRICE"、"SAVE"、"REDEEM" 等。
  每条写成 {{"label": 该行印的文字, "amount": 金额栏的数字取正}}。舍入行不算折扣。
  折扣常印在它所属商品的下一行，标签形如 "Buy 2 Save $6"、"MB APP UPGRADE -$10"、"5% OFF"。
  label 逐字照抄，amount 取金额栏。
- "subtotal"：SUBTOTAL / 小计 行，照原样。
- "tax"：税额行（TAX、PB1、VAT、GST 等），没有则填 0。
- "service"：服务费行（SERVICE、SVC 等），没有则填 0。
- "rounding"：舍入 / 调整行，保留正负号，没有则填 0。
- "total_paid"：实付行（TOTAL、CASH、OCTOPUS、VISA、CREDIT CARD、应付金额等），照原样。

规则：
- 只转写。不要加、减、核对或调平任何总额。
- 不要为了让账单对得上而改动任何数字。票面数字若看起来对不上，也照原样报。
- 一行一条，不合并、不编造、不遗漏。
- **数字连同千位分隔符和小数点一起照抄**。票面写 "60.000" 就写 "60.000"，
  写 "28,000" 就写 "28,000"，不要替你换算或改写格式。
- 只回一个 JSON 对象，不要别的内容：
{{"items": [], "discounts": [{{"label": "", "amount": 0}}], "subtotal": 0, "tax": 0, "service": 0, "rounding": 0, "total_paid": 0}}
"""

INSTRUCTION = "照抄这张票据的各行金额与合计，按字段定义输出 JSON。"


def image_data_url(path):
    """把本地图片编码成多模态消息可用的 data URL。"""
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    return f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode('ascii')}"


def pil_data_url(img, fmt="JPEG"):
    """CORD 的图像来自内存，不落盘也能编码。"""
    import io
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format=fmt, quality=92)
    return f"data:image/{fmt.lower()};base64,{base64.b64encode(buf.getvalue()).decode('ascii')}"


def build_chain(model=MODEL, temperature=0, timeout=180):
    from langchain_core.output_parsers import JsonOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_deepseek import ChatDeepSeek

    llm = ChatDeepSeek(model=model, temperature=temperature,
                       max_retries=3, timeout=timeout)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", [
            {"type": "text", "text": "{instruction}"},
            {"type": "image_url", "image_url": {"url": "{image_url}"}},
        ]),
    ])
    return prompt | llm | JsonOutputParser()


def read_many(chain, urls, reads=3, concurrency=CONCURRENCY):
    """对每张票据独立识别 reads 次，全部请求打包进一次并发批处理。

    返回 {票据下标: [原始 JSON, ...]}，失败的那次不计入。
    """
    payloads = [{"instruction": INSTRUCTION, "image_url": u} for _ in range(reads) for u in urls]
    owners = [i for _ in range(reads) for i in range(len(urls))]
    out = {i: [] for i in range(len(urls))}
    try:
        raw = chain.batch(payloads, config={"max_concurrency": concurrency},
                          return_exceptions=True)
    except Exception:
        return out
    for i, item in zip(owners, raw):
        if isinstance(item, dict):
            out[i].append(item)
    return out


In [ ]:
%%writefile /content/ghb/src/cord.py
"""读取 CORD (naver-clova-ix/cord-v2) 的标注，并折算成校验所需的量。

CORD 的 gt_parse 有三个超类：menu / sub_total / total。两处需要当心：
  - menu 只有一个商品时是 dict，多个时是 list
  - 金额写法不统一，交给 money.parse_amount 处理
"""
from decimal import Decimal

from money import parse_amount, q2

ZERO = Decimal("0.00")


def _as_list(node):
    """menu 可能是 dict（单品）或 list（多品），统一成 list。"""
    if node is None:
        return []
    return node if isinstance(node, list) else [node]


def _scalar(value):
    """同一字段被标注多次时值是 list（如 ['46.636', '46.636']），取第一个。"""
    while isinstance(value, list):
        if not value:
            return None
        value = value[0]
    return None if isinstance(value, dict) else value


def _sum(values):
    out = ZERO
    for v in values:
        if v is not None:
            out += v
    return out


def parse_gt(gt_parse, hint=None):
    """把一条 CORD 标注折算成校验需要的字段。"""
    menu = _as_list(gt_parse.get("menu"))
    sub = gt_parse.get("sub_total") or {}
    tot = gt_parse.get("total") or {}

    # 商品行：优先用 price（该行实收），没有就退回 unitprice*cnt 的近似 itemsubtotal
    item_prices, item_discounts = [], []
    for it in menu:
        if not isinstance(it, dict):
            continue
        p = parse_amount(_scalar(it.get("price")), hint)
        if p is None:
            p = parse_amount(_scalar(it.get("itemsubtotal")), hint)
        if p is not None:
            item_prices.append(p)
        d = parse_amount(_scalar(it.get("discountprice")), hint)
        if d is not None:
            item_discounts.append(abs(d))

    return {
        "items": item_prices,
        "items_total": _sum(item_prices),
        "item_discount": _sum(item_discounts),
        "subtotal": parse_amount(_scalar(sub.get("subtotal_price")), hint),
        "discount": abs(parse_amount(_scalar(sub.get("discount_price")), hint) or ZERO),
        "tax": parse_amount(_scalar(sub.get("tax_price")), hint) or ZERO,
        "service": parse_amount(_scalar(sub.get("service_price")), hint) or ZERO,
        "othersvc": parse_amount(_scalar(sub.get("othersvc_price")), hint) or ZERO,
        "total": parse_amount(_scalar(tot.get("total_price")), hint),
        "cash": parse_amount(_scalar(tot.get("cashprice")), hint),
        "change": parse_amount(_scalar(tot.get("changeprice")), hint),
        "n_items": len(item_prices),
    }


def check_items_explain_subtotal(rec, tol=Decimal("1")):
    """校验二：商品行总额能否解释小计。"""
    if rec["subtotal"] is None or not rec["items"]:
        return None
    gap = rec["items_total"] - rec["item_discount"] - rec["subtotal"]
    return gap if abs(gap) > tol else ZERO


def check_subtotal_explains_total(rec, tol=Decimal("1")):
    """校验一：小计加税费服务费减折扣，能否解释实付总额。"""
    if rec["subtotal"] is None or rec["total"] is None:
        return None
    expect = rec["subtotal"] + rec["tax"] + rec["service"] + rec["othersvc"] - rec["discount"]
    gap = rec["total"] - expect
    return gap if abs(gap) > tol else ZERO


def money_fields(gt_parse, hint=None):
    """列出这张票上所有可被篡改的金额字段路径与原值，供篡改实验挑选目标。"""
    out = []
    def add(path, value):
        for v in (value if isinstance(value, list) else [value]):
            if isinstance(v, str) and any(c.isdigit() for c in v):
                out.append((path, v))

    for i, it in enumerate(_as_list(gt_parse.get("menu"))):
        if isinstance(it, dict):
            add(f"menu[{i}].price", it.get("price"))
    for key in ("subtotal_price", "discount_price", "tax_price", "service_price"):
        add(f"sub_total.{key}", (gt_parse.get("sub_total") or {}).get(key))
    for key in ("total_price", "cashprice", "changeprice"):
        add(f"total.{key}", (gt_parse.get("total") or {}).get(key))
    return out


In [ ]:
%%writefile /content/ghb/src/tamper.py
"""图像篡改：把票面上的某个金额改掉，看核验能否发现。

两种方式，用途不同：

  render_over_box  整块重绘。按框高匹配字号，取该处背景色与墨色，把新数字画上去。
                   全自动，跨 100 张图稳定，用来跑 TPR / FPR 统计量。

  copy_glyph       字形复制。从同一张票上取同字体的数字，抽出墨迹覆盖度，
                   擦掉原字后按目标处的背景与墨色重新合成。视觉真实，
                   8 倍放大才看得出，用来做定性配图。

两种都只改图像，不碰标注——否则实验无效。
"""
import random
import re
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont

PAD = 4


# ---------------------------------------------------------------- 取色

def _light_median(patch):
    """背景色：取亮于中位数的像素的中位数，避开墨迹。"""
    gray = patch.mean(axis=2)
    light = patch[gray > np.median(gray)]
    return np.median(light, axis=0) if len(light) else np.median(patch.reshape(-1, 3), axis=0)


def _ink_color(patch):
    """墨色：最暗的 5% 像素的中位数。"""
    gray = patch.mean(axis=2)
    dark = patch[gray <= np.percentile(gray, 5)]
    return np.median(dark, axis=0) if len(dark) else np.array([60.0, 40.0, 40.0])


# ---------------------------------------------------------------- 改数字

def perturb_number(text, rng):
    """改动一个数字，保持字符串长度与分隔符不变，返回 (新串, 新值-旧值 的十进制位)。

    保持长度是为了重绘后仍能塞进原来的框。
    """
    digits = [i for i, c in enumerate(text) if c.isdigit()]
    if not digits:
        return None
    # 不改首位为 0，也不把唯一一位改成 0
    for _ in range(20):
        i = rng.choice(digits)
        old = text[i]
        new = rng.choice([d for d in "0123456789" if d != old])
        if i == digits[0] and new == "0" and len(digits) > 1:
            continue
        return text[:i] + new + text[i + 1:]
    return None


# ---------------------------------------------------------------- 方式一：整块重绘

def _font(size):
    for name in ("DejaVuSansMono-Bold.ttf", "DejaVuSansMono.ttf", "DejaVuSans-Bold.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    try:  # matplotlib 自带 DejaVu，Colab 上一定有
        import matplotlib
        p = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSansMono-Bold.ttf"
        return ImageFont.truetype(str(p), size)
    except Exception:
        return ImageFont.load_default()


def render_over_box(img, box, new_text):
    """把 box 区域擦掉并重绘 new_text，尽量贴合原来的字号与颜色。"""
    x0, y0, x1, y1 = [int(v) for v in box]
    arr = np.array(img.convert("RGB")).astype(float)
    h, w = arr.shape[:2]
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(w, x1), min(h, y1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return img, False

    ring = arr[max(0, y0 - PAD):y1 + PAD, max(0, x0 - PAD):x1 + PAD]
    bg, ink = _light_median(ring), _ink_color(arr[y0:y1, x0:x1])

    out = img.convert("RGB").copy()
    d = ImageDraw.Draw(out)
    d.rectangle([x0 - 1, y0 - 1, x1 + 1, y1 + 1], fill=tuple(int(v) for v in bg))

    # 字号按框高收敛，再按框宽微调，保证新数字不溢出
    size = max(8, int((y1 - y0) * 1.05))
    for _ in range(12):
        f = _font(size)
        tw, th = d.textbbox((0, 0), new_text, font=f)[2:]
        if tw <= (x1 - x0) and th <= (y1 - y0) * 1.25:
            break
        size -= 1
        if size < 8:
            break
    f = _font(size)
    tw, th = d.textbbox((0, 0), new_text, font=f)[2:]
    d.text((x1 - tw, y0 + ((y1 - y0) - th) // 2), new_text,
           font=f, fill=tuple(int(v) for v in ink))
    return out, True


# ---------------------------------------------------------------- 方式二：字形复制

def copy_glyph(arr, tgt, src, rng):
    """用 src 处的字形覆盖 tgt 处的字形，原地修改 arr（float RGB）。"""
    tx0, ty0, tx1, ty1 = tgt
    sx0, sy0, sx1, sy1 = src

    ring = arr[ty0 - PAD:ty1 + PAD, tx0 - PAD:tx1 + PAD]
    bg, ink = _light_median(ring), _ink_color(arr[ty0:ty1, tx0:tx1])
    noise = float(np.std(ring.mean(axis=2))) * 0.35

    ex0, ey0, ex1, ey1 = tx0 - PAD, ty0 - PAD, tx1 + PAD, ty1 + PAD
    hh, ww = ey1 - ey0, ex1 - ex0
    patch = np.broadcast_to(bg, (hh, ww, 3)) + rng.normal(0, noise, (hh, ww, 1))
    arr[ey0:ey1, ex0:ex1] = np.clip(patch, 0, 255)

    src_patch = arr[sy0:sy1, sx0:sx1]
    gray = src_patch.mean(axis=2)
    s_bg, s_ink = _light_median(src_patch).mean(), gray.min()
    alpha = np.clip((s_bg - gray) / max(s_bg - s_ink, 1e-6), 0, 1)[..., None]

    sh, sw = alpha.shape[:2]
    cy, cx = (ty0 + ty1) // 2, (tx0 + tx1) // 2
    py0, px0 = cy - sh // 2, cx - sw // 2
    region = arr[py0:py0 + sh, px0:px0 + sw]
    arr[py0:py0 + sh, px0:px0 + sw] = np.clip(region * (1 - alpha) + ink * alpha, 0, 255)


# ---------------------------------------------------------------- CORD 专用

def quad_to_box(quad):
    xs = [quad[f"x{i}"] for i in (1, 2, 3, 4)]
    ys = [quad[f"y{i}"] for i in (1, 2, 3, 4)]
    return min(xs), min(ys), max(xs), max(ys)


def find_word(valid_line, category, text):
    """在标注里找到某个类别下、文字等于 text 的词，返回它的框。

    gt_parse 里的值可能带货币前缀（"Rp 35.000"），而 OCR 把 "Rp" 和数字
    切成了两个词，所以还要按纯数字部分再找一轮。
    """
    want = text.strip()
    digits = re.sub(r"[^\d.,]", "", want).strip(".,")
    words = [(w, line) for line in valid_line
             if line.get("category") == category for w in line.get("words", [])]

    for w, _ in words:                              # 完全相等
        if w.get("text", "").strip() == want:
            return quad_to_box(w["quad"])
    for w, _ in words:                              # 词里包含整串
        if want and want in w.get("text", ""):
            return quad_to_box(w["quad"])
    if digits:                                      # 只比数字部分
        for w, _ in words:
            if re.sub(r"[^\d.,]", "", w.get("text", "")).strip(".,") == digits:
                return quad_to_box(w["quad"])
    return None


def field_to_category(field):
    """"menu[0].price" -> "menu.price"；标注里的 category 不带下标。"""
    return re.sub(r"\[\d+\]", "", field)


def tamper_cord(sample_image, valid_line, field, old_text, seed=0):
    """按字段名在 CORD 图像上改动一个金额。

    返回 (新图, 说明)；定位不到目标词时返回 (原图, None)。
    """
    rng = random.Random(seed)
    box = find_word(valid_line, field_to_category(field), old_text)
    if box is None:
        return sample_image, None
    new_text = perturb_number(old_text, rng)
    if not new_text or new_text == old_text:
        return sample_image, None
    out, ok = render_over_box(sample_image, box, new_text)
    if not ok:
        return sample_image, None
    return out, {"field": field, "old": old_text, "new": new_text, "box": box}


In [ ]:
import sys
sys.path.insert(0, "/content/ghb/src")
for m in ("money", "receipt", "chain", "cord", "tamper"):
    sys.modules.pop(m, None)
import money, receipt, chain, cord, tamper
print("源码已加载")

## 3. 载入 CORD 并做 E1

CORD-v2 的 test 划分共 100 张。先只看标注本身：两条算术校验各自适用多少张、
其中多少张自洽。**不自洽的那部分就是误报基线**——后面报 FPR 时要拿它作参照。

In [ ]:
import json
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2", split="test")
samples = []
for row in ds:
    gt = json.loads(row["ground_truth"])
    rec = cord.parse_gt(gt["gt_parse"])
    samples.append({"image": row["image"], "gt": gt, "rec": rec,
                    "c1": cord.check_subtotal_explains_total(rec),
                    "c2": cord.check_items_explain_subtotal(rec)})

both = [s for s in samples if s["c1"] is not None and s["c2"] is not None]
clean = [s for s in both if s["c1"] == 0 and s["c2"] == 0]
print(f"总计 {len(samples)} 张")
print(f"  校验一适用 {sum(s['c1'] is not None for s in samples)} 张，"
      f"自洽 {sum(s['c1'] == 0 for s in samples if s['c1'] is not None)} 张")
print(f"  校验二适用 {sum(s['c2'] is not None for s in samples)} 张，"
      f"自洽 {sum(s['c2'] == 0 for s in samples if s['c2'] is not None)} 张")
print(f"  两条都适用 {len(both)} 张，都自洽 {len(clean)} 张  <- 可用于篡改实验")
print(f"  标注层面就不自洽：{len(both) - len(clean)} 张（{(len(both)-len(clean))/max(len(both),1):.0%}）")

## 4. E2 字段抽取准确率

模型只做转写，金额在 Python 侧汇总，再与 CORD 标注逐字段比对。
`N_EVAL` 控制规模——每张要发 `READS` 次请求，先小后大。

In [ ]:
N_EVAL = 20      # 先跑 20 张确认流程，再放大
READS = 3

from decimal import Decimal
import time

pool = clean[:N_EVAL]
urls = [chain.pil_data_url(s["image"]) for s in pool]

t0 = time.time()
ch = chain.build_chain()
readings = chain.read_many(ch, urls, reads=READS)
print(f"{len(pool)} 张 x {READS} 次 = {len(pool)*READS} 次请求，耗时 {time.time()-t0:.0f}s")

fields = ["subtotal", "total_paid", "items_total"]
hit = {f: 0 for f in fields}
n_ok = 0
rows = []
for i, s in enumerate(pool):
    recs = [receipt.parse_reading(p, hint=".") for p in readings[i]]
    m = receipt.merge([r for r in recs if r], locale=receipt.IDR)
    if m is None:
        rows.append((i, "识别失败", None, None)); continue
    n_ok += 1
    want = {"subtotal": s["rec"]["subtotal"], "total_paid": s["rec"]["total"],
            "items_total": s["rec"]["items_total"]}
    for f in fields:
        if want[f] is not None and m[f] == want[f]:
            hit[f] += 1
    rows.append((i, m["subtotal"], m["total_paid"], want["subtotal"]))

print(f"\n有效识别 {n_ok}/{len(pool)} 张")
for f in fields:
    print(f"  {f:12s} 精确匹配 {hit[f]:3d}/{n_ok}  ({hit[f]/max(n_ok,1):.0%})")

## 5. E3 篡改检出

从两条校验都自洽的票据里取一批，随机一半做图像篡改（整块重绘，按框高匹配字号，
取该处背景与墨色），另一半保持原样。两组混在一起送检，统计：

- **TPR**：被改过的票里，判为存疑的比例
- **FPR**：没改过的票里，误判为存疑的比例

只报检出率是不诚实的，两个都要。

In [ ]:
import random
N_TAMPER = 20     # 一半改一半不改
rng = random.Random(42)

pool = clean[:N_TAMPER]
plan = []
for i, s in enumerate(pool):
    do_tamper = (i % 2 == 0)
    img, info = s["image"], None
    if do_tamper:
        targets = cord.money_fields(s["gt"]["gt_parse"])
        rng.shuffle(targets)
        for field, old in targets:
            img2, info = tamper.tamper_cord(s["image"], s["gt"]["valid_line"],
                                            field, old, seed=rng.randrange(10**6))
            if info:
                img = img2
                break
    plan.append({"idx": i, "tampered": bool(info), "image": img, "info": info})

made = sum(p["tampered"] for p in plan)
print(f"{len(plan)} 张里成功篡改 {made} 张，未改 {len(plan)-made} 张")
for p in plan[:6]:
    if p["info"]:
        print(f"  #{p['idx']}: {p['info']['field']}  {p['info']['old']} -> {p['info']['new']}")

In [ ]:
urls = [chain.pil_data_url(p["image"]) for p in plan]
t0 = time.time()
readings = chain.read_many(ch, urls, reads=READS)
print(f"{len(plan)} 张 x {READS} 次，耗时 {time.time()-t0:.0f}s\n")

tp = fp = tn = fn = 0
detail = []
for i, p in enumerate(plan):
    recs = [receipt.parse_reading(x, hint=".") for x in readings[i]]
    m = receipt.merge([r for r in recs if r], locale=receipt.IDR)
    if m is None:
        detail.append((p["idx"], p["tampered"], "识别失败", [])); continue
    v, failed, c = receipt.verdict(m, locale=receipt.IDR)
    flagged = (v == "存疑")
    if p["tampered"]:
        tp += flagged; fn += not flagged
    else:
        fp += flagged; tn += not flagged
    detail.append((p["idx"], p["tampered"], v, failed))

n_t, n_c = tp + fn, fp + tn
print(f"被篡改 {n_t} 张：检出 {tp}，漏检 {fn}   TPR = {tp/max(n_t,1):.0%}")
print(f"未篡改 {n_c} 张：误报 {fp}，正确 {tn}   FPR = {fp/max(n_c,1):.0%}")
print("\n逐张：")
for idx, t, v, failed in detail:
    print(f"  #{idx:3d}  {'已篡改' if t else '未篡改'}  判定={v:4s}  失败项={failed or '无'}")

## 6. 存结果

把三个实验的数字落盘，PPT 直接引用，不要凭记忆重打。

In [ ]:
import csv, json
out = Path("/content/ghb/results")
out.mkdir(exist_ok=True)

summary = {
    "E1_总数": len(samples), "E1_两条都适用": len(both), "E1_都自洽": len(clean),
    "E2_评测张数": len(clean[:N_EVAL]), "E2_有效识别": n_ok,
    "E2_字段命中": {f: hit[f] for f in fields},
    "E3_篡改张数": n_t, "E3_TPR": round(tp / max(n_t, 1), 4),
    "E3_未篡改张数": n_c, "E3_FPR": round(fp / max(n_c, 1), 4),
    "READS": READS,
}
(out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1),
                                  encoding="utf-8")
with (out / "e3_detail.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["idx", "tampered", "verdict", "failed"])
    for r in detail:
        w.writerow([r[0], r[1], r[2], "|".join(r[3])])
print(json.dumps(summary, ensure_ascii=False, indent=1))
print("\n已存至 /content/ghb/results/")

## 怎么用这些数字

- E1 的"标注层面就不自洽"比例，是误报的**下界**：这些票据连标注都对不上，
  系统判它存疑并非纯粹的错误
- E2 说明转写本身够不够准；若准确率低，E3 的 FPR 会被抬高
- E3 的 TPR / FPR 一起报。若 FPR 明显高于 E1 的基线，差值来自识别误差而非方法缺陷

**局限**：查的是票面算术是否自洽，不是像素取证。造假者若把商品行、小计、
付款行一并改成彼此吻合的数字，三条校验都会通过。本方法提高的是伪造成本。
